# ESnet → IRI Token Bootstrap Notebook

This notebook:
1. Authenticates against the ESnet OAuth2 endpoint using the **password grant**.
2. Decodes the returned JWT access token for inspection.
3. Writes the token `IRI_API_TOKEN` in the home directory `~/.iri_token.json`. 

## Required Environment Variables
Set these in your shell or `.env` file:

- `SENSE_AUTH_ENDPOINT`
- `SENSE_CLIENT_ID`
- `SENSE_SECRET`
- `SENSE_USERNAME`
- `SENSE_PASSWORD`
- Optional: `SENSE_VERIFY_TLS`, `SENSE_TIMEOUT`


## 1) Setup and Imports

In [ ]:
!pip install globus-sdk python-dotenv

In [ ]:
import os
import base64
import json
from datetime import datetime
from pathlib import Path

import requests
from dotenv import load_dotenv


## 2) Configuration (load environment variables)

In [ ]:
# Load environment variables from .env (if present)
load_dotenv()

AUTH_ENDPOINT = os.getenv("SENSE_AUTH_ENDPOINT")
CLIENT_ID = os.getenv("SENSE_CLIENT_ID")
SECRET = os.getenv("SENSE_SECRET")
USERNAME = os.getenv("SENSE_USERNAME")
PASSWORD = os.getenv("SENSE_PASSWORD")

VERIFY_TLS = os.getenv("SENSE_VERIFY_TLS", "true").lower() == "true"
TIMEOUT = int(os.getenv("SENSE_TIMEOUT", "30"))

if not all([AUTH_ENDPOINT, CLIENT_ID, SECRET, USERNAME, PASSWORD]):
    raise RuntimeError("Missing one or more required SENSE_* environment variables.")
print("Finished loading the environment file")

## 3) JWT decoding helpers

In [ ]:
def decode_jwt_part(part: str):
    part += "=" * (-len(part) % 4)
    return json.loads(base64.urlsafe_b64decode(part))


def decode_jwt(token: str):
    header, payload, signature = token.split(".")
    return decode_jwt_part(header), decode_jwt_part(payload)


## 4) Request access token

In [ ]:
data = {
    "grant_type": "password",
    "username": USERNAME,
    "password": PASSWORD,
    "scope": "offline_access",
}

response = requests.post(
    AUTH_ENDPOINT,
    data=data,
    verify=VERIFY_TLS,
    auth=(CLIENT_ID, SECRET),
    timeout=TIMEOUT,
)

print("HTTP status:", response.status_code)
print("\n=== Raw response ===")
print(response.text)

token = response.json()


## 5) Decode token information

In [ ]:
if "access_token" not in token:
    raise RuntimeError("No access_token received from auth endpoint.")

access_token = token["access_token"]

# Decode JWT for inspection
header, payload = decode_jwt(access_token)

print("\n=== JWT Header ===")
print(json.dumps(header, indent=2))

print("\n=== JWT Payload ===")
print(json.dumps(payload, indent=2))

if "iat" in payload:
    print("\nIssued at :", datetime.fromtimestamp(payload["iat"]))
if "exp" in payload:
    print("Expires at:", datetime.fromtimestamp(payload["exp"]))

## 6) Save Token for Other Notebooks

In [ ]:
home_path = Path("~/.iri_token.json").expanduser()

home_path.parent.mkdir(parents=True, exist_ok=True)

with open(home_path, "w") as f:
    json.dump({"IRI_API_TOKEN": access_token}, f)
print(f"Token saved: {home_path}")